In [0]:
%run ./00_config_uc


# 00 — Config (Unity Catalog + ADLS)

Centraliza catálogo/schema, storage account e paths base.


In [0]:
## Unity Catalog context (Azure)
spark.sql("USE CATALOG projeto_data_engineering")
spark.sql("CREATE SCHEMA IF NOT EXISTS kabum_project")
spark.sql("USE SCHEMA kabum_project")

# Storage account name (adjust if yours is different)
STORAGE_ACCOUNT = 'storagekabumdata'
BRONZE_ABFSS = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
SILVER_ABFSS = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
GOLD_ABFSS   = f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/"


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_TABLE = "projeto_data_engineering.kabum_project.products_bronze"
SILVER_TABLE = "projeto_data_engineering.kabum_project.products_clean"

df_raw = spark.read.table(BRONZE_TABLE)

# -------------------------
# Compatibilidade: se não vier com _src_file/paths, cria
# -------------------------
if '_src_file' not in df_raw.columns:
    df_raw = df_raw.withColumn('_src_file', F.lit(None).cast('string'))
if 'search_term_path' not in df_raw.columns:
    df_raw = df_raw.withColumn('search_term_path', F.lit(None).cast('string'))
if 'ingestion_date' not in df_raw.columns:
    df_raw = df_raw.withColumn('ingestion_date', F.current_date())
else:
    df_raw = df_raw.withColumn('ingestion_date', F.coalesce(F.col('ingestion_date'), F.current_date()))

def col_if_exists(df, name, cast_to=None):
    root = name.split('.')[0]
    if root in df.columns:
        c = F.col(name)
        return c.cast(cast_to) if cast_to else c
    return F.lit(None).cast(cast_to) if cast_to else F.lit(None)

def parse_brl_price(col_):
    s = F.trim(col_.cast('string'))
    s = F.regexp_replace(s, r"(?i)\b(de|por|à vista|pix|no pix|cart[aã]o|no cart[aã]o)\b[:\s]*", '')
    s = F.regexp_replace(s, r"(?i)r\$\s*", '')
    s = F.regexp_replace(s, r"\s+", '')
    s = F.regexp_replace(s, r"[^0-9\.,]", '')
    s = F.regexp_replace(s, r"\.(?=\d{3}(\D|$))", '')
    s = F.regexp_replace(s, ',', '.')
    return s.cast('double')

def normalize_discount_pct(col_):
    s = F.trim(col_.cast('string'))
    s = F.regexp_replace(s, '%', '')
    s = F.regexp_replace(s, r"[^0-9\.,]", '')
    s = F.regexp_replace(s, ',', '.')
    return s.cast('double')

# -------------------------
# BRAND: garantindo JSON válido
# -------------------------
df_raw = df_raw.withColumn(
    'brand_json',
    F.when(F.expr('typeof(brand)').startswith('struct'), F.to_json(F.col('brand')))
     .otherwise(col_if_exists(df_raw, 'brand', 'string'))
)

brand_id = F.expr("try_cast(get_json_object(brand_json, '$.id') as BIGINT)")
brand_name = F.get_json_object(F.col('brand_json'), '$.name')
brand_img = F.get_json_object(F.col('brand_json'), '$.img')

brand_id = F.coalesce(brand_id, F.expr(r"try_cast(regexp_extract(brand_json, '^\\s*(\\d+)', 1) as BIGINT)"))
brand_name = F.coalesce(
    brand_name,
    F.when(F.col('brand_json').contains(','), F.trim(F.element_at(F.split(F.col('brand_json'), ','), -1)))
     .otherwise(F.trim(F.col('brand_json')))
)
brand_name = F.trim(F.regexp_replace(brand_name, r"[{}\"]", ''))

price_raw_txt = F.coalesce(
    col_if_exists(df_raw, 'price', 'string'),
    col_if_exists(df_raw, 'price_text', 'string'),
    col_if_exists(df_raw, 'price_str', 'string'),
    col_if_exists(df_raw, 'price_formatted', 'string'),
    col_if_exists(df_raw, 'price_display', 'string'),
)
old_price_raw_txt = F.coalesce(
    col_if_exists(df_raw, 'old_price', 'string'),
    col_if_exists(df_raw, 'old_price_text', 'string'),
    col_if_exists(df_raw, 'old_price_str', 'string'),
    col_if_exists(df_raw, 'old_price_formatted', 'string'),
    col_if_exists(df_raw, 'old_price_display', 'string'),
)
discount_raw_txt = F.coalesce(
    col_if_exists(df_raw, 'discount_percentage', 'string'),
    col_if_exists(df_raw, 'discount_pct', 'string'),
    col_if_exists(df_raw, 'discount', 'string'),
)

df_silver = (
    df_raw
    .withColumn('marketplace', col_if_exists(df_raw, 'marketplace', 'string'))
    .withColumn('search_term', F.coalesce(col_if_exists(df_raw, 'search_term', 'string'), F.col('search_term_path')))
    .withColumn('product_url', col_if_exists(df_raw, 'product_url', 'string'))
    .withColumn('product_name', F.coalesce(col_if_exists(df_raw, 'product_name', 'string'), col_if_exists(df_raw, 'title', 'string')))
    .withColumn('brand_id', brand_id)
    .withColumn('brand', brand_name)
    .withColumn('brand_img', brand_img)
    .withColumn('category', col_if_exists(df_raw, 'category', 'string'))
    .withColumn('image_url', col_if_exists(df_raw, 'image', 'string'))
    .withColumn('currency', col_if_exists(df_raw, 'currency', 'string'))
    .withColumn('price_raw', price_raw_txt)
    .withColumn('old_price_raw', old_price_raw_txt)
    .withColumn('discount_pct_raw', discount_raw_txt)
    .withColumn('price', F.coalesce(F.when(col_if_exists(df_raw, 'price_value', 'double') > 0, col_if_exists(df_raw, 'price_value', 'double')), parse_brl_price(F.col('price_raw'))))
    .withColumn('old_price', F.coalesce(F.when(col_if_exists(df_raw, 'old_price_value', 'double') > 0, col_if_exists(df_raw, 'old_price_value', 'double')), parse_brl_price(F.col('old_price_raw'))))
    .withColumn('discount_pct_site', F.coalesce(F.when(col_if_exists(df_raw, 'discount_percentage', 'double') > 0, col_if_exists(df_raw, 'discount_percentage', 'double')), normalize_discount_pct(F.col('discount_pct_raw'))))
    .withColumn('rating', col_if_exists(df_raw, 'rating_value', 'double'))
    .withColumn('reviews_count', col_if_exists(df_raw, 'reviews_count', 'long'))
    .withColumn('kabum_code', col_if_exists(df_raw, 'kabum_code', 'long'))
    .withColumn('available', col_if_exists(df_raw, 'available', 'boolean'))
    .withColumn('max_installment', col_if_exists(df_raw, 'max_installment', 'string'))
    .withColumn('scraped_at', F.current_timestamp())
)

df_silver = df_silver.withColumn(
    'product_url',
    F.when(F.col('product_url').isNotNull(), F.regexp_replace(F.col('product_url'), r"(\?.*|#.*)$", '')).otherwise(F.col('product_url'))
)

df_silver = (
    df_silver
    .withColumn('price', F.when(F.col('price') > 0, F.col('price')).otherwise(F.lit(None).cast('double')))
    .withColumn('old_price', F.when(F.col('old_price') > 0, F.col('old_price')).otherwise(F.lit(None).cast('double')))
    .withColumn('old_price', F.when((F.col('old_price').isNotNull()) & (F.col('price').isNotNull()) & (F.col('old_price') > F.col('price')), F.col('old_price')).otherwise(F.lit(None).cast('double')))
    .withColumn('discount_pct', F.when((F.col('old_price').isNotNull()) & (F.col('price').isNotNull()) & (F.col('price') > 0) & (F.col('old_price') > F.col('price')), F.round((F.col('old_price') - F.col('price')) / F.col('old_price') * 100).cast('int')).otherwise(F.lit(0)))
    .withColumn('discount_pct_gap', F.when(F.col('discount_pct_site').isNotNull(), (F.col('discount_pct_site') - F.col('discount_pct')).cast('double')).otherwise(F.lit(None).cast('double')))
)

df_silver = df_silver.filter((F.col('product_url').isNotNull()) | ((F.col('product_name').isNotNull()) & (F.col('brand').isNotNull())))

df_silver = df_silver.withColumn(
    'product_key',
    F.sha2(F.coalesce(F.col('product_url'), F.concat_ws('||', F.coalesce(F.col('product_name'), F.lit('')), F.coalesce(F.col('brand'), F.lit('')))), 256)
)

w = Window.partitionBy('product_key', 'ingestion_date', 'search_term').orderBy(F.col('scraped_at').desc())
df_silver = df_silver.withColumn('_rn', F.row_number().over(w)).filter(F.col('_rn') == 1).drop('_rn', 'brand_json')

final_cols = [
    'ingestion_date','marketplace','search_term','product_key','kabum_code','product_name',
    'brand_id','brand','brand_img','category','available','currency','price','old_price','discount_pct',
    'rating','reviews_count','max_installment','product_url','image_url','scraped_at',
    'price_raw','old_price_raw','discount_pct_raw','discount_pct_site','discount_pct_gap'
]
final_cols = [c for c in final_cols if c in df_silver.columns]
df_silver_final = df_silver.select(*final_cols)

# Overwrite dinâmico: sobrescreve somente as partições presentes no df_silver_final
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

(
  df_silver_final.write
    .format("delta")
    .mode("overwrite")
    # NÃO usar overwriteSchema com dynamic partition overwrite
    .partitionBy("ingestion_date", "search_term")
    .saveAsTable(SILVER_TABLE)
)


In [0]:
%sql
SELECT
  COUNT(*) AS rows,
  SUM(CASE WHEN price IS NOT NULL THEN 1 ELSE 0 END) AS price_filled,
  SUM(CASE WHEN old_price IS NOT NULL THEN 1 ELSE 0 END) AS old_price_filled,
  SUM(CASE WHEN discount_pct IS NOT NULL THEN 1 ELSE 0 END) AS discount_pct_filled,
  MIN(price) AS min_price,
  MAX(price) AS max_price
FROM kabum_project.silver.products_clean;

In [0]:
%sql
-- =========================================================
-- ✅ Validação para ver se tabela esta vazia
-- =========================================================
SELECT
  COUNT(*) AS rows,
  MIN(price) AS min_price,
  MAX(price) AS max_price
FROM kabum_project.silver.products_clean;


In [0]:
%sql
CREATE TABLE IF NOT EXISTS projeto_data_engineering.kabum_project.products_clean
USING DELTA
LOCATION 'abfss://silver@storagekabumdata.dfs.core.windows.net/kabum/products_clean'